# 01.6 — Sentence Segmentation

**Problem it solves:** Most NLP tasks operate on sentences, not whole documents. Splitting a document into sentences correctly is harder than it looks.

**Why it's hard:** Periods are ambiguous. 'Mr. Smith went to Washington. He loved it.' — two sentences, two periods, but only one is a boundary.

---

## Part 1: Why Naive Splitting Fails

In [ ]:
hard_cases = """
Dr. Smith earned his Ph.D. at M.I.T. in the U.S.A.
The price is $3.50 per lb. That's very reasonable.
She said, "I can't believe it!" He nodded.
Visit www.example.com. It has all the info.
Error: NullPointerException at line 42. Stack trace follows.
The meeting is Jan. 5th at 3 p.m. Please be on time.
He asked: is this the end? Yes. No. Maybe.
""".strip()

# Naive: split on any . ! ?
import re
naive = [s.strip() for s in re.split(r'[.!?]', hard_cases) if s.strip()]
print("NAIVE SPLIT (wrong):")
for i, s in enumerate(naive, 1):
    print(f"  [{i:2d}] {s}")

## Part 2: Punkt-Style Unsupervised Sentence Segmenter

Kiss & Strunk (2006) — learns abbreviations and sentence starters from the corpus itself. This is what NLTK's `sent_tokenize` uses.

In [ ]:
from collections import Counter, defaultdict
import re, math

class PunktSentenceSegmenter:
    """
    Simplified Punkt algorithm.
    Core insight: a period following a short or capitalized token is probably
    an abbreviation, not a sentence boundary.
    """
    
    def __init__(self):
        self.abbrev_types = set()
        self.sent_starters = set()
    
    def _get_period_context(self, text: str) -> list[dict]:
        """Find all period tokens and their context."""
        contexts = []
        tokens = text.split()
        for i, tok in enumerate(tokens):
            if '.' in tok:
                before = tokens[i-1] if i > 0 else ''
                after = tokens[i+1] if i < len(tokens) - 1 else ''
                word = re.sub(r'[^a-zA-Z.]', '', tok)
                contexts.append({
                    'token': tok, 'word': word,
                    'before': before, 'after': after,
                    'pos': i,
                })
        return contexts
    
    def train(self, text: str):
        """Learn abbreviations from corpus."""
        # Heuristic: short tokens ending in period are abbreviations
        # Words with internal periods are abbreviations (U.S.A.)
        tokens = text.split()
        
        type_counts = Counter()
        period_counts = Counter()
        
        for tok in tokens:
            # Strip trailing punctuation for counting
            word = tok.rstrip('.,;:!?').lower()
            if word:
                type_counts[word] += 1
            if tok.endswith('.'):
                word_no_period = tok[:-1].lower()
                period_counts[word_no_period] += 1
        
        # Classify as abbreviation if:
        # 1. Short (<=5 chars) OR contains internal period
        # 2. Appears with period frequently
        for word, freq in period_counts.items():
            total = type_counts.get(word, 0) + freq
            is_short = len(word) <= 4
            has_internal_period = '.' in word
            mostly_with_period = freq / total > 0.5 if total > 0 else False
            
            if (is_short or has_internal_period) and mostly_with_period:
                self.abbrev_types.add(word)
        
        return self
    
    def _is_boundary(self, before_token: str, after_token: str) -> bool:
        """Decide if period after before_token is a sentence boundary."""
        word = re.sub(r'[^a-zA-Z.]', '', before_token).lower().rstrip('.')
        
        # Known abbreviation -> NOT a boundary
        if word in self.abbrev_types:
            return False
        
        # Short uppercase token (initials) -> NOT a boundary
        if len(word) <= 2 and word.isupper():
            return False
        
        # Next token starts with uppercase -> likely boundary
        if after_token and after_token[0].isupper():
            return True
        
        # Next token starts with lowercase -> likely continuation
        if after_token and after_token[0].islower():
            return False
        
        return True  # default: treat as boundary
    
    def segment(self, text: str) -> list[str]:
        """Split text into sentences."""
        # Split on whitespace, preserving the tokens
        tokens = text.split()
        sentences = []
        current = []
        
        for i, tok in enumerate(tokens):
            current.append(tok)
            
            # Does this token end with sentence-final punctuation?
            if re.search(r'[.!?]["\']?$', tok):
                after = tokens[i+1] if i < len(tokens) - 1 else ''
                
                if tok.endswith('.'):
                    if self._is_boundary(tok, after):
                        sentences.append(' '.join(current))
                        current = []
                else:  # ! or ? are almost always boundaries
                    sentences.append(' '.join(current))
                    current = []
        
        if current:
            sentences.append(' '.join(current))
        
        return [s.strip() for s in sentences if s.strip()]

# Train on a sample corpus
train_corpus = """
Dr. Smith and Prof. Jones met at 3 p.m. on Jan. 5th.
The U.S.A. is a country. So is the U.K.
Mr. Brown works at Inc. Corp. He is very busy.
The cat sat. The dog ran. They played.
"""

segmenter = PunktSentenceSegmenter()
segmenter.train(train_corpus)
print(f"Learned abbreviations: {sorted(segmenter.abbrev_types)}")

print("\n=== PUNKT SEGMENTATION ===")
sentences = segmenter.segment(hard_cases)
for i, s in enumerate(sentences, 1):
    print(f"  [{i:2d}] {s}")

## Part 3: Evaluation — How Good Is Your Segmenter?

In [ ]:
def evaluate_segmenter(predicted: list[str], gold: list[str]) -> dict:
    """Measure precision, recall, F1 at sentence boundary level."""
    # Convert to boundary positions (cumulative char positions)
    def get_boundaries(sentences):
        boundaries = set()
        pos = 0
        for s in sentences[:-1]:  # no boundary after last sentence
            pos += len(s)
            boundaries.add(pos)
        return boundaries
    
    pred_b = get_boundaries(predicted)
    gold_b = get_boundaries(gold)
    
    tp = len(pred_b & gold_b)
    fp = len(pred_b - gold_b)
    fn = len(gold_b - pred_b)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {'precision': precision, 'recall': recall, 'f1': f1, 
            'tp': tp, 'fp': fp, 'fn': fn}

# Gold standard
gold = [
    "Dr. Smith earned his Ph.D. at M.I.T. in the U.S.A.",
    "The price is $3.50 per lb.",
    "That's very reasonable.",
    "She said, \"I can't believe it!\"",
    "He nodded.",
]

# Naive segmenter output (for comparison)
naive_segs = [s.strip() + '.' for s in re.split(r'\.\s+', 
    'Dr. Smith earned his Ph.D. at M.I.T. in the U.S.A. The price is $3.50 per lb. That\'s very reasonable.') if s.strip()]

punkt_segs = segmenter.segment(
    'Dr. Smith earned his Ph.D. at M.I.T. in the U.S.A. '
    'The price is $3.50 per lb. That\'s very reasonable.'
)

print("Punkt segments:", punkt_segs)
metrics = evaluate_segmenter(punkt_segs, gold[:3])
print(f"\nMetrics: {metrics}")

## Summary

**What it does:** Splits a document into sentence units for downstream processing.

**Approaches ranked by robustness:**
1. Naive period-split — breaks constantly
2. Rule-based with abbreviation list — works for clean text, brittle on new domains
3. Punkt (unsupervised) — learns from data, much better, still imperfect
4. Neural (Spacy, stanza) — best accuracy, especially for noisy/social media text

**What breaks all of them:**
- Code snippets, file paths, URLs with internal periods
- Bullet points and lists (no sentence-ending punctuation)
- Legal/financial documents with unusual formatting
- Languages without clear word boundaries (Thai, Chinese need different approach)

---
**Module 01 complete. Next: Module 02 — Classical Retrieval, starting with the Inverted Index.**